# StepCounter with Python and Pandas

### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

### Setting constants

In [ ]:
factor_std = 2.122 # factor to multiply the standard deviation for threshold calculation
factor_peak = 0.68 # factor to multiply the peak value for threshold calculation

start_time = 4 # start time for filtering the dataframe
end_time = 60 # end time for filtering the dataframe


### Setting renderer for export

In [ ]:
pio.renderers.default = "notebook_connected"

### Change working directory for interpreter

In [ ]:
working_directory = r"D:\Your\Path\Schrittzähler"

os.chdir(working_directory)

### Read data from .csv file

In [ ]:
df = pd.read_csv("100steps.csv", encoding = "utf-8") # assign data to Pandas Dataframe
df.head()

### Rename columns

In [ ]:
df.rename(
    columns= {
        "Time (s)": "time_s",
        "Acceleration x (m/s^2)": "acc_x",
        "Acceleration y (m/s^2)": "acc_y",
        "Acceleration z (m/s^2)": "acc_z",
        "Absolute acceleration (m/s^2)": "acc_abs"
    },
    inplace = True
) # rename columns for better readability

df.head()

### Plot values

In [ ]:
# plot acceleration in x, y and z direction over time
px.line(df, x = "time_s", y = ["acc_x", "acc_y", "acc_z"], title = "Acceleration in x, y and z direction over time", height = 400, width = 1200)

### Calculate total acceleration and append to Pandas Dataframe

In [ ]:
# add calculated total acceleration from x, y and z acceleration to dataframe
df["acc_total"] = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2)
df.head()

### Plot values

In [ ]:
# plot absolute acceleration over time
px.line(df, x = "time_s", y = "acc_abs", title = "Absolute acceleration over time", height = 600, width = 1200)

### Prepare Dataframe for calculations

In [ ]:
# filter dataframe for time between start_time and end_time
df_filtered = df[(df["time_s"] > start_time) & (df["time_s"] < end_time)]

### Define threshold by percentage

In [ ]:
# add threshold to dataframe
threshold = df_filtered["acc_total"].max() * factor_peak

# add threshold to dataframe
df_filtered["threshold"] = threshold

### Define threshold by standard deviation

In [ ]:
# calculate threshold based on mean and standard deviation
mean = df_filtered["acc_total"].mean()
std = df_filtered["acc_total"].std()
threshold_std = mean + factor_std * std

# add threshold to dataframe
df_filtered["threshold_std"] = threshold_std

### Histogram

In [ ]:
# plot histogram of total acceleration for filtered dataframe
px.histogram(df_filtered, x = "acc_total", title = "Distribution of total acceleration (filtered)", height = 400, width = 1200)

### Plot filtered data with calculated thresholds

In [ ]:
# plot absolute acceleration over time for filtered dataframe
px.line(df_filtered, x = "time_s", y = ["acc_total", "threshold", "threshold_std"], title = "Absolute acceleration over time (filtered)", height = 600, width = 1200)

### Values above thresholds

In [ ]:
# filter dataframe for values above threshold
df_above_threshold = df_filtered[df_filtered["acc_total"] > threshold]

# filter dataframe for values above threshold based on mean and standard deviation
df_above_threshold_std = df_filtered[df_filtered["acc_total"] > threshold_std]

### Plot values above thresholds

In [ ]:
# plot total acceleration over time for values above threshold
px.line(df_above_threshold, x = "time_s", y = "acc_total", title = "Total acceleration over time (above threshold)", height = 300, width = 1200)


In [ ]:
# plot total acceleration over time for values above threshold based on mean and standard deviation
px.line(df_above_threshold_std, x = "time_s", y = "acc_total", title = "Total acceleration over time (above threshold based on mean and standard deviation)", height = 300, width = 1200)


### Flag values above thresholds

In [ ]:
# create flag for values above threshold
above_threshold = df_filtered["acc_total"] > threshold
# create flag for values above threshold based on mean and standard deviation
above_threshold_std = df_filtered["acc_total"] > threshold_std

# add flag to dataframe for values above threshold
df_filtered["above_threshold"] = above_threshold
# add flag to dataframe for values above threshold based on mean and standard deviation
df_filtered["above_threshold_std"] = above_threshold_std

### Plot flags

In [ ]:
# plot thresholds over time for filtered dataframe
px.line(df_filtered, x = "time_s", y = "above_threshold", title = "Thresholds over time (filtered)", height = 300, width = 1200)

In [ ]:
# plot thresholds based on mean and standard deviation over time for filtered dataframe
px.line(df_filtered, x = "time_s", y = "above_threshold_std", title = "Thresholds based on mean and standard deviation over time (filtered)", height = 300, width = 1200)

### Counting steps

In [ ]:
# count number of steps based on flag for values above threshold
df_filtered["above_threshold"].diff().sum() / 2

In [ ]:
# count number of steps based on flag for values above threshold based on mean and standard deviation
df_filtered["above_threshold_std"].diff().sum() / 2

### Export to html

In [ ]:
%%cmd
jupyter nbconvert --to html step_counter.ipynb